# Tutorial 06: Releases Dataset Conversion

Convert RTD from vintage format to releases format.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent))

from peru_gdp_rtd.config import get_settings
from peru_gdp_rtd.transformers import convert_to_releases_dataset

settings = get_settings('../config/config.yaml')

## Vintage vs Releases Format

**Vintage Format (RTD)**:
```
vintage_id | 2020_01 | 2020_02 | 2020_03
2020_02    |   3.5   |   2.1   |   NaN
2020_03    |   3.3   |   2.0   |   1.8
```

**Releases Format**:
```
target_period | first_release | second_release | third_release
2020_01       |     3.5       |      3.3       |      3.4
2020_02       |     2.1       |      2.0       |      1.9
```

## Step 1: Load RTD

In [ ]:
rtd_path = Path('../data/output/monthly_gdp_rtd.csv')
if rtd_path.exists():
    monthly_rtd = pd.read_csv(rtd_path, index_col=0)
    print(f'RTD shape: {monthly_rtd.shape}')
    display(monthly_rtd.head())
else:
    print('RTD file not found. Run pipeline first.')

## Step 2: Convert to Releases Format

In [ ]:
if 'monthly_rtd' in locals():
    releases = convert_to_releases_dataset(monthly_rtd)
    print(f'Releases shape: {releases.shape}')
    display(releases.head())

## Step 3: Analyze Revisions

In [ ]:
if 'releases' in locals():
    # Calculate revision (second - first release)
    releases['revision_1to2'] = releases['second_release'] - releases['first_release']
    
    print('Revision Statistics:')
    print(releases['revision_1to2'].describe())
    
    # Plot revision distribution
    import matplotlib.pyplot as plt
    releases['revision_1to2'].hist(bins=30)
    plt.xlabel('Revision (percentage points)')
    plt.ylabel('Frequency')
    plt.title('Distribution of GDP Revisions')
    plt.show()

## Step 4: Save Releases Dataset

In [ ]:
if 'releases' in locals():
    output_path = Path('../data/output/monthly_gdp_releases.csv')
    releases.to_csv(output_path)
    print(f'Saved to: {output_path}')

## Use Cases for Releases Format

1. **Revision Analysis**: Study patterns of data revisions
2. **Nowcasting**: Predict final values from early releases
3. **Rationality Tests**: Test if revisions are forecastable
4. **Policy Analysis**: Understand real-time data constraints

## Next Steps

- Run complete pipeline: `python scripts/update_rtd.py`
- Explore the release-format outputs with the examples in this notebook
- Read documentation: `docs/USAGE.md`